# 레지스트리·다이제스트와 클라우드 배포

태그와 다이제스트가 가리키는 배포 대상을 비교합니다.

### 이 준비 셀이 하는 일

첫 셀의 `Path` 탐색은 어느 상위 폴더에서 열어도 Ch33의 `pyproject.toml`을 기준으로 실습 폴더를 찾습니다. 뒤의 `subprocess.run(..., cwd=lab, capture_output=True)`은 명령을 별도 프로세스로 실행하고 화면 문구와 종료 코드를 함께 반환합니다. 이 보조 코드는 CI를 흉내 내는 공통 실행 경계이며, 각 절의 판정은 다음 셀에서 해당 명령의 실제 반환값을 읽어 결정합니다.

In [1]:
from pathlib import Path
import hashlib, subprocess, sys, tempfile, json
lab = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "ch33-cicd" and (p / "pyproject.toml").is_file()), Path("labs/part7/ch33-cicd")).resolve()

### 바로 다음 코드를 읽는 방법

이미지 태그는 이동 가능한 이름입니다. 노트북의 `docker image inspect ... {{.Id}}`는 레지스트리 manifest digest가 아니라 현재 Docker 데몬의 로컬 이미지 ID를 반환합니다. 실습은 v1과 v2 이미지의 로컬 ID가 다른지 확인하고, 교육용 JSON의 `digest` 필드에 v1의 로컬 ID를 저장합니다. 필드 이름만 보고 레지스트리 매니페스트를 검증했다고 해석하지 않습니다.

In [2]:
for revision in ("v1", "v2"):
    built = subprocess.run(["docker", "build", "-q", "-t", f"ch33-demo:{revision}", "--build-arg", f"APP_REVISION={revision}", "."], cwd=lab, text=True, capture_output=True)
    print(revision, built.stdout.strip()[:27], "exit_code=", built.returncode)
    assert built.returncode == 0
digest_v1 = subprocess.check_output(["docker", "image", "inspect", "ch33-demo:v1", "--format", "{{.Id}}"], text=True).strip()
digest_v2 = subprocess.check_output(["docker", "image", "inspect", "ch33-demo:v2", "--format", "{{.Id}}"], text=True).strip()
print("same_tag=latest", "same_digest=", digest_v1 == digest_v2)
assert digest_v1 != digest_v2

v1 sha256:1cb76db0ea58f43f50dc exit_code= 0


v2 sha256:4067958c02de6da69345 exit_code= 0
same_tag=latest same_digest= False


### 이 단계에서 구분할 범위

특정 이미지 식별자를 기록해도 그 이미지가 안전하거나 올바르다는 보장은 아닙니다. 서명·취약점 검사·테스트 결과를 같은 산출물과 연결해야 합니다. 레지스트리 manifest digest는 실제 push 뒤 별도로 기록하며, 이번 로컬 실험에서는 관찰하지 않았습니다. 클라우드 배포 권한과 환경 승인은 이미지 생성 권한과 분리합니다.

In [3]:
with tempfile.TemporaryDirectory() as temp:
    t = Path(temp)
    (t / "scripts").mkdir()
    (t / "runs").mkdir()
    (t / "scripts/deploy.py").write_text((lab / "scripts/deploy.py").read_text())
    result = subprocess.run([sys.executable, "scripts/deploy.py", digest_v1], cwd=t, text=True, capture_output=True)
    print(result.stdout.strip(), "exit_code=", result.returncode)
    print((t / "runs/deployment.json").read_text().strip())
    assert result.returncode == 0

deployed revision=1 digest=sha256:1cb76db0ea58... exit_code= 0
[
  {
    "revision": 1,
    "event": "deploy",
    "digest": "sha256:1cb76db0ea58f43f50dc84141dfc2896efed098aa2e67e01fc9f5116cc417328",
    "status": "ready"
  }
]


## 결과 해석

같은 latest 태그라는 설명만으로 실제 바이트를 식별할 수 없지만, 내용에서 계산한 다이제스트는 v1과 v2가 달랐습니다. 로컬 배포 기록은 다이제스트를 저장했습니다.